# ThinkGuard — full experiment (Google Colab)

**Runtime:** GPU (**Runtime → Change runtime type → T4 / L4 / A100**).

1. Publish **your** submission repo to GitHub (not `luka-group/ThinkGuard`).  
2. Set `REPO_URL` below to that repo’s **HTTPS clone URL**.  
3. Add Colab secret **`HF_TOKEN`** (read token; enable notebook access). Needed for gated **`allenai/wildguardmix`** and any gated checkpoints.

Artifacts appear under `/content/ThinkGuard-work/results/` — zip + download at the end.

In [ ]:
# --- Configuration ---
REPO_URL = "https://github.com/ishraqsadik/ThinkGuard-repro-submission.git"
BRANCH = "main"

# Sample caps (raise/remove caps for full submission runs)
MAX_SAMPLES_BEAVER = 500
MAX_SAMPLES_TOXIC = 500
MAX_SAMPLES_WILDGUARD = 300
MAX_SAMPLES_OPENAI = None  # None = all (~1680 rows)

MAX_SAMPLES_REFLECT_WILD = 128
LATENCY_PROMPTS = 16
LATENCY_REPEATS = 40

SMOKE_ONLY = False  # True: skip heavy experiment cells

import os
try:
    from google.colab import userdata

    if "HF_TOKEN" not in os.environ:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except ImportError:
    pass

assert os.environ.get("HF_TOKEN"), "Set Colab secret HF_TOKEN or export HF_TOKEN before running."

In [ ]:
!nvidia-smi
import torch

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
%%capture cap_install
import pathlib
import shutil
import subprocess

WORK = pathlib.Path("/content/ThinkGuard-work")
if WORK.exists():
    shutil.rmtree(WORK)
subprocess.run(
    ["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(WORK)],
    check=True,
)
%cd /content/ThinkGuard-work

!pip install -q -r requirements.txt
!pip install -q git+https://github.com/meta-llama/llama-cookbook.git

import sys

sys.path.insert(0, "/content/ThinkGuard-work/src")
print("Ready:", WORK.resolve())

In [ ]:
from tg_eval.lg3_imports import load_prompt_format_utils

print("Loaded", load_prompt_format_utils().__name__)

## Experiment 1 — four benchmarks

Runs `scripts/run_eval.py` once per dataset so each can use its own `--max-samples`.
Writes JSON summaries + `metrics_summary.csv` fragments under each output directory.

In [ ]:
import os
import subprocess

os.chdir("/content/ThinkGuard-work")
env = os.environ.copy()


def run(cmd: list[str]) -> None:
    print(" ".join(cmd))
    subprocess.run(cmd, env=env, check=True)


if not SMOKE_ONLY:
    base = [
        "python",
        "scripts/run_eval.py",
        "--max-new-tokens",
        "512",
    ]
    run(base + ["--benchmark", "beaver", "--max-samples", str(MAX_SAMPLES_BEAVER), "--output-dir", "results/beaver"])
    run(base + ["--benchmark", "toxic", "--max-samples", str(MAX_SAMPLES_TOXIC), "--output-dir", "results/toxic"])
    run(base + ["--benchmark", "wildguard", "--max-samples", str(MAX_SAMPLES_WILDGUARD), "--output-dir", "results/wildguard"])
    oai_cmd = base + ["--benchmark", "openai", "--output-dir", "results/openai"]
    if MAX_SAMPLES_OPENAI is not None:
        oai_cmd += ["--max-samples", str(MAX_SAMPLES_OPENAI)]
    run(oai_cmd)
else:
    print("SMOKE_ONLY=True — skipped")

## Experiment 2 — Extension A (reflective WildGuardMix)

Adds `summary_reflect.json` + `reflect_vs_single.png`.

In [ ]:
import os
import subprocess

os.chdir("/content/ThinkGuard-work")
if not SMOKE_ONLY:
    subprocess.run(
        [
            "python",
            "scripts/run_eval.py",
            "--benchmark",
            "wildguard",
            "--reflect",
            "--max-samples",
            str(MAX_SAMPLES_REFLECT_WILD),
            "--output-dir",
            "results/wildguard_reflect",
            "--max-new-tokens",
            "512",
        ],
        env=os.environ.copy(),
        check=True,
    )
else:
    print("SMOKE_ONLY=True — skipped")

## Experiment 3 — Extension B (latency + NF4 subset accuracy)

**Latency** loads FP16/BF16 then NF4 **sequentially** — restart runtime if VRAM errors occur.
**NF4 accuracy** reruns BeaverTails with `--quantize-4bit` on a fixed subset.

In [ ]:
import os
import subprocess

os.chdir("/content/ThinkGuard-work")
if not SMOKE_ONLY:
    subprocess.run(
        [
            "python",
            "scripts/bench_latency.py",
            "--benchmark",
            "beaver",
            "--num-prompts",
            str(LATENCY_PROMPTS),
            "--repeats",
            str(LATENCY_REPEATS),
            "--max-new-tokens",
            "64",
            "--output-json",
            "results/latency.json",
        ],
        check=True,
    )
    subprocess.run(
        [
            "python",
            "scripts/run_eval.py",
            "--benchmark",
            "beaver",
            "--max-samples",
            "300",
            "--output-dir",
            "results/beaver_fp16",
            "--max-new-tokens",
            "512",
        ],
        check=True,
    )
    subprocess.run(
        [
            "python",
            "scripts/run_eval.py",
            "--benchmark",
            "beaver",
            "--max-samples",
            "300",
            "--output-dir",
            "results/beaver_nf4",
            "--max-new-tokens",
            "512",
            "--quantize-4bit",
        ],
        check=True,
    )
else:
    print("SMOKE_ONLY=True — skipped")

## Download `results/` as a zip

If `files.download` fails in your browser, use the Colab **Files** pane (`/content/thinkguard_results.zip`).

In [ ]:
!cd /content/ThinkGuard-work && zip -r /content/thinkguard_results.zip results/

try:
    from google.colab import files

    files.download("/content/thinkguard_results.zip")
except ImportError:
    print("Not on Colab — zip saved to /content/thinkguard_results.zip")